# PDF markdown samples & LLM extraction

Browse the markdown `classify_stored_pdfs()` extracted into `pdf_analysis.extracted_markdown_pages`,
and run `llm_client.extract_project_profile` against real samples to check output and iterate on the prompt.

Requirements:
- `docker compose up -d postgres` running (this notebook reads pdf_analysis via `docker compose exec postgres psql`,
  since the DB port isn't published to the host).
- For the LLM cells: set `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT` in `.env` at the repo root, and start
  the kernel from the repo root: `settings()` reads `.env` relative to the working directory.

Extraction cells use the v3 schema: `extract_project_profile` returns an `ExtractionOutcome` whose
`document` holds one profile per offer. For scored runs over the labeled evaluation set, use
`scripts/eval_project_profiles.py` (see `docs/pdf-project-profile-v3-plan.md`).


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/home/haris/git/tum_ps_portal")
sys.path.insert(0, str(REPO_ROOT / "backend"))

from IPython.display import Markdown, display


def run_psql(sql: str) -> list[dict]:
    """Runs SQL against the live DB through the running postgres container (no host port is published)."""
    result = subprocess.run(
        ["docker", "compose", "exec", "-T", "postgres", "psql", "-U", "portal", "-d", "portal", "-A", "-t", "-c", sql],
        cwd=REPO_ROOT, capture_output=True, text=True, check=True,
    )
    return [json.loads(line) for line in result.stdout.splitlines() if line.strip()]


def fetch_samples(classification: str | None = None, limit: int = 3) -> list[dict]:
    where = f"WHERE pa.classification = '{classification}'" if classification else ""
    sql = f"""
        SELECT row_to_json(t) FROM (
            SELECT c.slug AS chair, pdf.url, pa.content_hash, pa.classification,
                   pa.classifier_version, pa.extracted_markdown_pages AS pages
            FROM pdf_analysis pa
            JOIN pdf_artifacts pdf ON pdf.content_hash = pa.content_hash
            JOIN listings l ON l.normalized->>'artifact_url' = pdf.url
            JOIN chairs c ON c.id = l.chair_id
            {where}
            ORDER BY random()
            LIMIT {limit}
        ) t;
    """
    return run_psql(sql)


def show_sample(sample: dict, page_limit: int = 1) -> None:
    header = (f"### {sample['chair']}\n\n"
              f"**classification:** {sample['classification']} ({sample['classifier_version']})  \n"
              f"**url:** {sample['url']}  \n"
              f"**content_hash:** `{sample['content_hash']}`")
    display(Markdown(header))
    for i, page in enumerate(sample["pages"][:page_limit], start=1):
        display(Markdown(f"---\n**page {i}**\n\n" + (page or "*(empty)*")))


## Browse extracted markdown by classification

Re-run a cell to see a different random sample.

In [ ]:
for sample in fetch_samples("digital_native", limit=2):
    show_sample(sample, page_limit=1)


In [ ]:
for sample in fetch_samples("mixed", limit=3):
    show_sample(sample, page_limit=3)


In [ ]:
for sample in fetch_samples("scanned", limit=1):
    show_sample(sample, page_limit=1)


## Run llm_client extraction and inspect the ProjectProfile output

These calls hit Azure OpenAI directly (no DB connection needed — markdown pages come from the
`fetch_samples()` cells above), so they work from the host as long as `.env` has
`AZURE_OPENAI_API_KEY` / `AZURE_OPENAI_ENDPOINT` set. `extract_project_profile_for_pdf` (which reads
from the DB directly instead) will NOT work from this notebook — it needs to run inside a container
that can resolve the `postgres` hostname.


In [ ]:
from app.ingestion import llm_client
from app.ingestion.pdf_reader import render_for_llm


def show_profile(profile) -> None:
    display(Markdown(f"```json\n{profile.model_dump_json(indent=2)}\n```"))


In [ ]:
sample = fetch_samples("digital_native", limit=1)[0]
markdown_text = render_for_llm(sample["pages"])
show_sample(sample, page_limit=1)

try:
    outcome = llm_client.extract_project_profile(markdown_text)
    show_profile(outcome.document)
except RuntimeError as error:
    print(f"Azure OpenAI not configured yet: {error}")
except llm_client.ProfileExtractionError as error:
    print(f"Extraction failed after {error.attempts} attempts: {error.errors}")


### Tweak the prompt

Edit `custom_prompt` below and re-run — no need to touch `llm_client.py` or restart the kernel
between tries. Once you're happy with a variant, copy it back into `SYSTEM_PROMPT` in
`backend/app/ingestion/llm_client.py`.


In [ ]:
custom_prompt = llm_client.SYSTEM_PROMPT + """

Additional instruction: <edit me>
"""

try:
    outcome = llm_client.extract_project_profile(markdown_text, system_prompt=custom_prompt)
    show_profile(outcome.document)
except RuntimeError as error:
    print(f"Azure OpenAI not configured yet: {error}")
except llm_client.ProfileExtractionError as error:
    print(f"Extraction failed after {error.attempts} attempts: {error.errors}")


## Reproducible extraction review

Use the same three PDFs for every prompt or schema variant: a German Project Study, an English
technical IDP, and an IDP with a blank extracted second page. This avoids duplicate PDFs from
the random browsing query above. `status=partial_text` means a missing fact may be on an unreadable
page; it must not become a negative filter claim.

The English `SYSTEM_PROMPT` handles German and English source documents. It preserves free-text
values in the source language and returns `document_language` separately from working language.
Review each list item and its own source quote before treating it as a search or filter fact.


In [ ]:
import re
from app.ingestion import llm_client
from app.ingestion.pdf_reader import render_for_llm
from app.ingestion.profile_evidence import check_profile_evidence
from app.ingestion.project_profile import PdfTextCoverage, SCHEMA_VERSION

FIXED_SAMPLES = [
    ("German Project Study", "885bf62fcae9b2b363115f1a8e448c2defa6afeae151417a3b387ea5dc97dae4"),
    ("English technical IDP", "010f18ee6272c81bfdf9871208afbf78c01786ce69886701676cf70f41197f3f"),
    ("Mixed IDP", "14ae909fda2b7aa67c33cd3eb06f46fbdd8c1b2dbc2020f757f74bf6d8f8737f"),
]


def fetch_sample_by_hash(content_hash: str) -> dict:
    if not re.fullmatch(r"[0-9a-f]{64}", content_hash):
        raise ValueError("Expected a SHA-256 content hash")
    rows = run_psql(f"""SELECT row_to_json(t) FROM (
        SELECT content_hash, classification, classifier_version, extracted_markdown_pages AS pages
        FROM pdf_analysis WHERE content_hash = '{content_hash}'
    ) t;""")
    if len(rows) != 1:
        raise ValueError(f"Expected one analysis row for {content_hash}")
    return rows[0]


for label, content_hash in FIXED_SAMPLES:
    sample = fetch_sample_by_hash(content_hash)
    coverage = PdfTextCoverage.from_pages(sample["classification"], sample["classifier_version"], sample["pages"])
    print(label, content_hash[:8], coverage.status, "pages", coverage.page_count,
          "blank", coverage.blank_markdown_pages)


Select one fixed PDF, run the extraction, and inspect the per-item evidence and flagged quotes.
The citation check removes Markdown/HTML formatting before matching, but its flags still need
human review. For prompt comparisons, keep the sample and schema fixed and record missed or
unsupported facts, especially required versus recommended skills and inferred deliverables.


In [ ]:
SAMPLE_INDEX = 0  # 0 German Project Study, 1 English IDP, 2 mixed IDP
label, content_hash = FIXED_SAMPLES[SAMPLE_INDEX]
sample = fetch_sample_by_hash(content_hash)
result = llm_client.extract_project_profile_result_for_pages(
    content_hash, sample["pages"], sample["classification"], sample["classifier_version"])
print(label, "schema", result.schema_version, "status", result.status, "attempts", result.attempts,
      "coverage", result.coverage.status, "short", result.coverage.short_text,
      "tokens in/out", result.input_tokens, result.output_tokens, "errors", result.errors)
if result.document:
    print("language", result.document.document_language, "kind", result.document.document_kind,
          "offers", len(result.document.offers))
    for index, offer in enumerate(result.document.offers):
        item_count = sum(len(getattr(offer, name).items) for name in type(offer).model_fields
                         if hasattr(getattr(offer, name), "items"))
        print(index, offer.title.value, "pages", offer.source_pages, "sourced items", item_count)
print("citation issues", [issue.model_dump() for issue in result.evidence_issues])
show_profile(result)
